<a href="https://colab.research.google.com/github/LionPaul/retro-game-miner/blob/main/notebooks/02_silver_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os
import re

# ==============================================================================
# PIPELINE CONFIGURATION
# ==============================================================================
CONSOLE_NAME = "snes"

DATASETS_DIR = "datasets"
# AJUSTE: Nomes adaptados para seguir estritamente o padrão numérico global do projeto
INPUT_FILENAME = f"1_{CONSOLE_NAME}_games_raw.xlsx"        # Lendo o arquivo gerado pelo Código 1
OUTPUT_FILENAME = f"2_{CONSOLE_NAME}_games_cleaned.xlsx"    # Gerando a base número 2

INPUT_PATH = os.path.join(DATASETS_DIR, INPUT_FILENAME)
OUTPUT_PATH = os.path.join(DATASETS_DIR, OUTPUT_FILENAME)
# ==============================================================================

def pipeline_silver_pentatendativa(input_path, output_path):
    """
    Cleans Wikipedia data and generates 5 strategic variations for API scraping.
    Acts as the Silver Layer of the Medallion Architecture.
    """
    print(f"🧹 Starting PENTATENDATIVA processing for Silver Layer: {CONSOLE_NAME.upper()}")

    if not os.path.exists(input_path):
        print(f"❌ Error: Raw file '{input_path}' not found. Please run Notebook 01 first.")
        return None

    df = pd.read_excel(input_path)

    # 1. Padronização de Colunas Dinâmica (Agnóstica a consoles)
    mapeamento_colunas = {}
    for col in df.columns:
        col_clean = col.lower()
        if 'title' in col_clean: mapeamento_colunas[col] = 'Title'
        elif 'developer' in col_clean: mapeamento_colunas[col] = 'Developer'
        elif 'publisher' in col_clean: mapeamento_colunas[col] = 'Publisher'
        elif 'japan' in col_clean: mapeamento_colunas[col] = 'Release_Japan'
        elif 'north america' in col_clean or 'na' in col_clean: mapeamento_colunas[col] = 'Release_NA'
        elif 'pal region' in col_clean or 'pal' in col_clean: mapeamento_colunas[col] = 'Release_PAL'

    df = df.rename(columns=mapeamento_colunas)

    # 2. Filtro Regional (Mantém apenas o mercado que nos importa: América do Norte)
    if 'Release_NA' in df.columns:
        df = df[~df['Release_NA'].astype(str).str.contains('Unreleased', na=False, case=False)]

    # 3. Consolidação de Datas (Calcula o lançamento original mais antigo do planeta)
    colunas_datas = ['Release_Japan', 'Release_NA', 'Release_PAL']
    colunas_presentes = [c for c in colunas_datas if c in df.columns]
    if colunas_presentes:
        for col in colunas_presentes:
            df[col] = df[col].astype(str).str.replace(r'\[.*\]', '', regex=True)
        datas_convertidas = df[colunas_presentes].apply(pd.to_datetime, errors='coerce')
        df['Original_Release'] = datas_convertidas.min(axis=1).dt.strftime('%Y-%m-%d')
        df = df.drop(columns=colunas_presentes, errors='ignore')

    # 4. GERADOR CRIATIVO DE 5 ESTRATÉGIAS DE BUSCA
    def gerar_5_estrategias_busca(nome_completo):
        nome = str(nome_completo)

        # Limpeza universal de colchetes e parênteses da Wiki
        nome = re.sub(r'\[.*\]|\(.*\)', '', nome)
        partes = nome.split('•')

        # --- ESTRATÉGIA 1: Nome Principal Limpo ---
        nome_p = partes[0]
        nome_p = re.sub(r'\d{4}-\d{2}-\d{2}.*|\d{4}.*', '', nome_p) # Arranca anos soltos
        nome_p = re.sub(r'(PAL|JP|NA|USA|JPN|MX)$', '', nome_p, flags=re.IGNORECASE).strip()
        nome_p = nome_p.rstrip(':-— ,.')
        title_1 = nome_p if len(nome_p) > 1 else None

        # --- ESTRATÉGIA 2: Wiki Fallback (Nome Regional) ---
        title_2 = None
        if len(partes) > 1:
            nome_alt = partes[1]
            nome_alt = re.sub(r'\d{4}-\d{2}-\d{2}.*|\d{4}.*', '', nome_alt)
            nome_alt = re.sub(r'(PAL|JP|NA|USA|JPN|MX)$', '', nome_alt, flags=re.IGNORECASE).strip()
            nome_alt = nome_alt.rstrip(':-— ,.')
            if len(nome_alt) > 1:
                title_2 = nome_alt

        # Quebra as palavras para as próximas estratégias
        palavras = nome_p.split()

        # --- ESTRATÉGIA 3: Keyword Title (Busca Ampla) ---
        title_3 = None
        if len(palavras) > 1:
            title_3 = " ".join(palavras[:2])
            # Se for termo muito comum, expande para 3 palavras
            if title_3.lower() in ['the', 'super', 'mega', 'a', 'an', 'legend of'] and len(palavras) > 2:
                title_3 = " ".join(palavras[:3])

        # --- ESTRATÉGIA 4: Acronym Title (Iniciais/Siglas) ---
        title_4 = None
        if len(palavras) >= 3:
            sigla = "".join([palavra[0] for word in palavras if (palavra := re.sub(r'\W+', '', word))])
            if len(sigla) >= 3:
                title_4 = sigla.upper()

        # --- ESTRATÉGIA 5: No Stopwords (Sem artigos/conjunções) ---
        title_5 = None
        stopwords = ['the', 'of', 'and', 'to', 'a', 'an', 'in', 'for', 'on']
        palavras_filtradas = [w for w in palavras if w.lower() not in stopwords]
        if len(palavras_filtradas) < len(palavras) and len(palavras_filtradas) > 0:
            title_5 = " ".join(palavras_filtradas)

        return pd.Series([title_1, title_2, title_3, title_4, title_5])

    # Aplica o gerador de 5 vertentes
    df[['T1_Clean', 'T2_Fallback', 'T3_Keyword', 'T4_Acronym', 'T5_NoStopwords']] = df['Title'].apply(gerar_5_estrategias_busca)

    # Organização das colunas e descarte das informações irrelevantes para a análise
    colunas_finais = [
        'Title', 'T1_Clean', 'T2_Fallback', 'T3_Keyword', 'T4_Acronym', 'T5_NoStopwords',
        'Original_Release', 'Developer', 'Publisher'
    ]
    df = df[colunas_finais]

    # Persistência dos dados higienizados
    df.to_excel(output_path, index=False)
    print(f"💾 Silver Layer successfully saved at: {output_path}")

    # Retorna o preview analítico
    return df[['Title', 'T1_Clean', 'T2_Fallback', 'T3_Keyword', 'T4_Acronym', 'T5_NoStopwords']].head(10)

# Executa o pipeline de limpeza
df_silver_preview = pipeline_silver_pentatendativa(INPUT_PATH, OUTPUT_PATH)